# 06 - Evaluation: all three tiers on all 15 questions

This is where the project's result comes from. Every question goes through
Tier 1, Tier 2 and Tier 3, and we score them the same way.

The two gaps are what we are actually measuring:

- **Tier 1 to Tier 2** - what using real data is worth
- **Tier 2 to Tier 3** - what checking is worth

Takes a while on a laptop - 15 questions x 3 tiers, and Tier 3 retries.


## Setup


In [ ]:
import sys
sys.path.append('../src')

import yaml
import pandas as pd
import matplotlib.pyplot as plt

import evidenceiq as eiq

questions = yaml.safe_load(open('../eval/benchmark.yaml'))['questions']
print(len(questions), 'questions')


## 1. Run everything


In [ ]:
rows = []

for i, q in enumerate(questions, 1):
    for tier, fn in [(1, eiq.tier1), (2, eiq.tier2), (3, eiq.tier3)]:
        print('[%2d/15] %s tier %d' % (i, q['id'], tier), end=' ')
        try:
            a = fn(q['question'])
            s = eiq.score(a)
            rows.append({
                'id': q['id'],
                'tier': tier,
                'difficulty': q['difficulty'],
                'answerable': q['answerable'],
                'gt_value': q.get('gt_value'),
                'values': [c.get('value') for c in a['claims']
                           if c.get('value') is not None],
                'refused': a['insufficient_data'],
                'claims': s['claims'],
                'supported': s['supported'],
                'unsupported_rate': s['unsupported_rate'],
                'evidence_coverage': s['evidence_coverage'],
                'queries_ok': s['queries_ok'],
                'retries': a['retries'],
                'seconds': a['seconds'],
            })
            print('ok')
        except Exception as e:
            print('ERROR', e)

df = pd.DataFrame(rows)
df.to_csv('../eval/results/all_runs.csv', index=False)
print('saved', len(df), 'rows')


## 2. Was the answer right?

A question counts as correct if any number the tier gave is within 1% of
the answer we worked out ourselves in SQL.


In [ ]:
def is_correct(row):
    if row['gt_value'] is None or pd.isna(row['gt_value']):
        return None                     # the trick questions have no number
    for v in row['values']:
        if eiq.about_equal(float(v), float(row['gt_value'])):
            return 1.0
    return 0.0


df['correct'] = df.apply(is_correct, axis=1)

answerable = df[df.answerable].copy()
tricks = df[~df.answerable].copy()

df[['id', 'tier', 'gt_value', 'values', 'correct']].head(12)


## 3. The scoreboard

These are the nine things the project asks us to measure.


In [ ]:
summary = pd.DataFrame({
    'accuracy':          answerable.groupby('tier').correct.mean(),
    'queries_ok':        df.groupby('tier').queries_ok.mean(),
    'evidence_coverage': df.groupby('tier').evidence_coverage.mean(),
    'unsupported_rate':  df.groupby('tier').unsupported_rate.mean(),
    'refused_tricks':    tricks.groupby('tier').refused.sum(),
    'avg_retries':       df.groupby('tier').retries.mean(),
    'avg_seconds':       df.groupby('tier').seconds.mean(),
}).round(3)

summary.to_csv('../eval/results/summary.csv')
summary


## 4. Accuracy by difficulty


In [ ]:
(answerable.pivot_table(index='difficulty', columns='tier',
                       values='correct', aggfunc='mean')
           .reindex(['easy', 'medium', 'hard'])
           .round(2))


## 5. The three impossible questions

The clearest table in the whole project. Anything that is not a refusal is
a made-up number a manager would have acted on.


In [ ]:
tricks.pivot_table(index='id', columns='tier', values='refused')


## 6. Charts for the report


In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 3.5))

answerable.groupby('tier').correct.mean().plot(
    kind='bar', ax=ax[0], color='steelblue')
ax[0].set_title('Accuracy')
ax[0].set_ylim(0, 1)

df.groupby('tier').unsupported_rate.mean().plot(
    kind='bar', ax=ax[1], color='indianred')
ax[1].set_title('Made-up claims (lower is better)')

df.groupby('tier').seconds.mean().plot(
    kind='bar', ax=ax[2], color='grey')
ax[2].set_title('Seconds per answer')

for a in ax:
    a.set_xlabel('tier')

plt.tight_layout()
plt.savefig('../eval/results/comparison.png', dpi=150)
plt.show()


## 7. What this shows

Fill this in once you have the real numbers:

- Tier 1 to Tier 2: accuracy went from ___ to ___ (that is what using the
  real data is worth)
- Tier 2 to Tier 3: made-up claims went from ___ to ___ (that is what the
  checking is worth)
- Tier 3 took ___ seconds against Tier 2's ___

**Say the cost out loud.** "Slower but much more accurate" is a believable
finding. Pretending there is no trade-off is not.
